# IMPLEMENTATION OF TCN-BiLSTM ARCHITECTURE ON XJTU DATASET

This notebook demonstrates the training and validation of the TCN-BiLSTM-Attention architecture across multiple window sizes on the XJTU-SY bearing dataset.

In [ ]:
import os
import glob
import time
import gc
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler

import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import Custom Architecture
import sys
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from importlib import import_module
tcn_bilstm_module = import_module('TCN-BiLSTM_Implementation')
TCN_BiLSTM_Attention = tcn_bilstm_module.TCN_BiLSTM_Attention

# Matplotlib Publication Standards
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'lines.linewidth': 2.0,
    'grid.alpha': 0.3
})
sns.set_style("whitegrid")



## 1. Global Configurations
Setup paths, experiment variables, and model hyperparameters.

In [ ]:
# Dataset Paths
DATASET_DIR = r"D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset"
RESULTS_DIR = r"D:\Proyek Dosen\Riset Bearing\TCN_BiLSTM_Results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Experiment Setup
WINDOW_SIZES = [30, 40, 50, 70]
NUM_FEATURES = 15 # Based on sliding window pipeline

# Training Hyperparameters
EPOCHS = 200
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_HEADS = 4

print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")


Num GPUs Available: 0


## 2. Helper Functions
Metrics, plotting, and evaluation utilities.

In [ ]:
def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def relative_prediction_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def health_index_to_rul(hi, max_rul):
    return np.clip(hi, 0, 1) * max_rul

def asymmetric_loss_scoring_function(y_pred, y_true, a=10, b=13):
    y_pred = tf.convert_to_tensor(y_pred, dtype=tf.float32)
    y_true = tf.convert_to_tensor(y_true, dtype=tf.float32)
    diff = y_pred - y_true
    
    loss_early = tf.exp(-diff / a) - 1.0
    loss_late = tf.exp(diff / b) - 1.0
    loss = tf.where(diff < 0, loss_early, loss_late)
    return tf.reduce_mean(loss).numpy()

def plot_health_index_prediction(y_true, y_pred, title, turning_point_minute=None,
                                  time_axis=None, save_path=None):
    """
    Plots expected vs predicted RUL on the absolute-minute X-axis.
    Args:
        time_axis (np.ndarray): Absolute minute values from Original_Minute.
        turning_point_minute (int): Absolute minute for the CUSUM change point line.
    """
    x = time_axis if time_axis is not None else np.arange(len(y_true))
    plt.figure(figsize=(12, 6))
    plt.plot(x, y_true, label='Expected RUL', color='black', linestyle='-')
    plt.plot(x, y_pred, label='Predicted RUL', color='gold', linestyle='--')
    
    if turning_point_minute is not None:
        plt.axvline(x=turning_point_minute, color='red', linestyle='-', alpha=0.7,
                    label=f'Turning Point @ Min {turning_point_minute}')
        
    plt.title(title, fontweight='bold')
    plt.xlabel('Absolute Time (Minutes)')
    plt.ylabel('Remaining Useful Life (RUL)')
    plt.legend(loc='upper right')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

## 3. Training & Evaluation Pipeline
Looping over specified window sizes to train models and evaluate on validation bearings.

In [ ]:
# Summary Storage
summary_metrics = []
# For saving predictions across all experiments into an Excel file
excel_writer_path = os.path.join(RESULTS_DIR, "TCN_BiLSTM_Predictions_Summary.xlsx")
excel_predictions = {} 

# Load Metadata to extract Total Lifespans (Tf) and Turning Points (Tcp)
metadata_file = os.path.join(DATASET_DIR, "bearing_metadata.csv")
if os.path.exists(metadata_file):
    metadata_df = pd.read_csv(metadata_file)
else:
    print(f"[FATAL ERROR] Metadata file not found at {metadata_file}. Cannot proceed.")
    sys.exit(1)

for ws in WINDOW_SIZES:
    print(f"\n{'='*60}\nStarting Experiment for Window Size: {ws}\n{'='*60}")
    
    ws_res_dir = os.path.join(RESULTS_DIR, f"window_size_{ws}")
    os.makedirs(ws_res_dir, exist_ok=True)
    
    # ----------------------------------------------------
    # A. LOAD TRAINING DATA
    # ----------------------------------------------------
    train_file = os.path.join(DATASET_DIR, f"processed_train_w{ws}.csv")
    if not os.path.exists(train_file):
        print(f"WARNING: Train file {train_file} not found. Skipping...")
        continue
        
    print(f"Loading training data: {train_file}")
    df_train = pd.read_csv(train_file)
    
    if df_train.empty:
        print(f"WARNING: Training dataset for Window Size {ws} is empty. Skipping...")
        continue
    
    y_train_scaled = df_train['Health_Index'].values
    
    # Target Deep Sanitization
    if np.isnan(y_train_scaled).any() or np.isinf(y_train_scaled).any():
        print(f"[DATA LOG] Target HI contains NaN/Inf! Applying nan_to_num recovery.")
        y_train_scaled = np.nan_to_num(y_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Exclude metadata columns
    drop_cols = ['Health_Index', 'Target_RUL']
    if 'Bearing_ID' in df_train.columns: drop_cols.append('Bearing_ID')
    if 'Change_Point' in df_train.columns: drop_cols.append('Change_Point')
    if 'Original_Minute' in df_train.columns: drop_cols.append('Original_Minute')
        
    X_train_flat = df_train.drop(columns=drop_cols).values
    if np.isnan(X_train_flat).any() or np.isinf(X_train_flat).any():
        print(f"[DATA LOG] Features contain NaN/Inf! Applying nan_to_num recovery.")
        X_train_flat = np.nan_to_num(X_train_flat, nan=0.0, posinf=0.0, neginf=0.0)
        
    # RobustScaler: immune to extreme vibration spikes
    feature_scaler = RobustScaler()
    X_train_flat = feature_scaler.fit_transform(X_train_flat)
    
    num_samples = X_train_flat.shape[0]
    num_features = X_train_flat.shape[1] // ws
    X_train_3d = X_train_flat.reshape(num_samples, ws, num_features)
    
    # Free memory
    del df_train, X_train_flat
    gc.collect()
    
    # ----------------------------------------------------
    # B. INITIALIZE MODEL & TRAIN (TCN-BiLSTM)
    # ----------------------------------------------------
    tf.keras.backend.clear_session()
    
    model_wrapper = TCN_BiLSTM_Attention(
        window_size=ws,
        num_features=num_features,
        num_heads=NUM_HEADS
    )
    
    print("Training Model...")
    
    # Using Early Stopping Callback
    early_stop_cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )
    
    start_train_time = time.time()
    
    history = model_wrapper.model.fit(
        X_train_3d, y_train_scaled,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[early_stop_cb],
        verbose=0  # set to 1 or 2 for detailed output
    )
    
    train_duration = time.time() - start_train_time
    print(f"Training phase finished in {train_duration:.2f} seconds.")
    
    model_wrapper.model.save(os.path.join(ws_res_dir, "best_tcn_bilstm_model.h5"))
    
    # Plot Training Loss
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Train MSE Loss', color='red')
    plt.plot(history.history['val_loss'], label='Val MSE Loss', color='blue')
    plt.title(f'Training Loss History (Window Size = {ws})')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(ws_res_dir, "training_loss_history.png"), dpi=300)
    plt.close()
    
    # Free Memory
    del X_train_3d, y_train_scaled
    gc.collect()
    tf.keras.backend.clear_session()
    
    # ----------------------------------------------------
    # C. EVALUATION ON VALIDATION BEARINGS
    # ----------------------------------------------------
    val_files = glob.glob(os.path.join(DATASET_DIR, f"processed_val_*_w{ws}.csv"))
    
    ws_predictions_df = pd.DataFrame()
    ws_metrics = []
    
    for v_file in val_files:
        b_match = re.search(r"processed_val_(.*)_w\d+\.csv", os.path.basename(v_file))
        b_id = b_match.group(1) if b_match else 'Unknown'
        print(f"Evaluating on {b_id} (Window={ws})...")
        
        df_val = pd.read_csv(v_file)
        
        if df_val.empty:
            print(f"WARNING: Validation dataset for {b_id} (Window Size {ws}) is empty. Skipping metric calculation...")
            continue
            
        # Load Metadata Configuration for Target Unscaling
        b_meta = metadata_df[metadata_df['Bearing_ID'] == b_id]
        if not b_meta.empty:
            b_tf = float(b_meta['Total_Lifespan'].iloc[0])
            b_tcp = int(b_meta['Change_Point'].iloc[0])
        else:
            print(f"[WARN] Meta info not found for {b_id}, using fallback constants.")
            b_tf = df_val['Target_RUL'].max()
            b_tcp = 0

        y_val_true_hi = df_val['Health_Index'].values
        y_val_true_rul = df_val['Target_RUL'].values
        
        val_drop_cols = ['Health_Index', 'Target_RUL']
        if 'Bearing_ID' in df_val.columns: val_drop_cols.append('Bearing_ID')
        if 'Change_Point' in df_val.columns: val_drop_cols.append('Change_Point')
        if 'Original_Minute' in df_val.columns: val_drop_cols.append('Original_Minute')
            
        X_val_flat = df_val.drop(columns=val_drop_cols).values
        X_val_flat = np.nan_to_num(X_val_flat, nan=0.0, posinf=0.0, neginf=0.0)
        X_val_flat = feature_scaler.transform(X_val_flat)
        
        ns = X_val_flat.shape[0]
        nf = X_val_flat.shape[1] // ws
        X_val_3d = X_val_flat.reshape(ns, ws, nf)
        
        # Predict
        start_infer_time = time.time()
        y_val_pred_hi = model_wrapper.model.predict(X_val_3d, batch_size=1024, verbose=0).flatten()
        infer_duration = time.time() - start_infer_time
        
        y_val_pred_hi = np.nan_to_num(y_val_pred_hi, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Reconstruct RUL limits
        y_val_pred_rul = health_index_to_rul(y_val_pred_hi, b_tf)
        
        # Metrics
        rmse_val = rmse_score(y_val_true_rul, y_val_pred_rul)
        mae_val = mean_absolute_error(y_val_true_rul, y_val_pred_rul)
        r2_val = r2_score(y_val_true_rul, y_val_pred_rul)
        rpe_val = relative_prediction_error(y_val_true_rul, y_val_pred_rul)
        asym_loss = asymmetric_loss_scoring_function(y_val_pred_rul, y_val_true_rul)
        
        ws_metrics.append({
            'Bearing_ID': b_id,
            'Window_Size': ws,
            'Training_Time(s)': train_duration,
            'Inference_Time(s)': infer_duration,
            'RMSE': rmse_val,
            'MAE': mae_val,
            'R2': r2_val,
            'RPE(%)': rpe_val,
            'Asymmetric_Loss': asym_loss
        })
        
        if 'Original_Minute' in df_val.columns:
            time_steps = df_val['Original_Minute'].values
        else:
            time_steps = np.arange(len(y_val_true_hi))
            
        # DataFrame
        b_pred_df = pd.DataFrame({
            'Bearing_ID': b_id,
            'Time_Step': time_steps,
            'Expected_Health_Index': y_val_true_hi,
            'Predicted_Health_Index': y_val_pred_hi,
            'Expected_Remaining_RUL': y_val_true_rul,
            'Predicted_Remaining_RUL': y_val_pred_rul
        })
        ws_predictions_df = pd.concat([ws_predictions_df, b_pred_df], ignore_index=True)
        
        # Plot on absolute minute axis
        plot_title = f"{b_id} RUL Prediction (TCN-BiLSTM - WS={ws})"
        plot_save_path = os.path.join(ws_res_dir, f"prediction_plot_{b_id}.png")
        plot_health_index_prediction(y_val_true_rul, y_val_pred_rul, plot_title,
                                      turning_point_minute=b_tcp, time_axis=time_steps,
                                      save_path=plot_save_path)
        
        del df_val, X_val_flat, X_val_3d
        gc.collect()

    if not ws_predictions_df.empty:
        ws_predictions_df.to_csv(os.path.join(ws_res_dir, f"prediction_states_w{ws}.csv"), index=False)
        excel_predictions[f"WS_{ws}"] = ws_predictions_df
        
    if len(ws_metrics) > 0:
        metrics_df = pd.DataFrame(ws_metrics)
        metrics_df.to_csv(os.path.join(ws_res_dir, f"evaluation_markers_w{ws}.csv"), index=False)
        
        avg_metrics = metrics_df.mean(numeric_only=True)
        summary_metrics.append({
            'Window_Size': ws,
            'Mean_RMSE': avg_metrics['RMSE'],
            'Mean_MAE': avg_metrics['MAE'],
            'Mean_R2': avg_metrics['R2'],
            'Mean_RPE(%)': avg_metrics['RPE(%)'],
            'Mean_Asymmetric_Loss': avg_metrics['Asymmetric_Loss']
        })

print(f"\n[INFO] Validating Metric Export Buffer...")
if len(excel_predictions) > 0:
    with pd.ExcelWriter(excel_writer_path) as writer:
        for sheet_name, df_sheet in excel_predictions.items():
            df_sheet.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f">> Successfully Converted to Master Excel Format: {excel_writer_path}")
else:
    print(">> Processing Halt: No numeric matrices survived network processing.")


Starting Experiment for Window Size: 30
Loading training data: D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset\processed_train_w30.csv

Training Model...
Restoring model weights from the end of the best epoch: 200.



Starting Experiment for Window Size: 40
Loading training data: D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset\processed_train_w40.csv
Training Model...
Epoch 16: early stopping
Restoring model weights from the end of the best epoch: 1.



Starting Experiment for Window Size: 50
Loading training data: D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset\processed_train_w50.csv
Training Model...
Epoch 112: early stopping
Restoring model weights from the end of the best epoch: 97.



Starting Experiment for Window Size: 70
Loading training data: D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset\processed_train_w70.csv
Training Model...
Restoring model weights from the end of the best epoch: 196.



Saving all predictions to summary Excel: D:\Proyek Dosen\Riset Bearing\TCN_BiLSTM_Results\TCN_BiLSTM_Predictions_Summary.xlsx
All Experiments Completed Succesfully!


## 4. Final Window Size Optimization Analysis
Comparing validating mean MSE/RMSE across the 4 window size variants to deduce the optimal length.

In [5]:
summary_df = pd.DataFrame(summary_metrics)

if not summary_df.empty:
    display(summary_df)
    summary_df.to_csv(os.path.join(RESULTS_DIR, "TCN_BiLSTM_WindowSize_Comparison.csv"), index=False)
    
    plt.figure(figsize=(10, 6))
    plt.plot(summary_df['Window_Size'], summary_df['Mean_RMSE'], marker='o', linestyle='-', color='purple', linewidth=2.5, markersize=8)
    
    # Annotate points
    for i, row in summary_df.iterrows():
        plt.annotate(f"{row['Mean_RMSE']:.4f}", 
                     (row['Window_Size'], row['Mean_RMSE']),
                     textcoords="offset points", 
                     xytext=(0,10), 
                     ha='center')
                     
    plt.title('Mean Validation RMSE vs Window Size (TCN-BiLSTM)', fontweight='bold', pad=15)
    plt.xlabel('Window Size')
    plt.ylabel('Mean Validation RMSE')
    plt.xticks(summary_df['Window_Size'].values)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "optim_tcn_bilstm_rmse_vs_windowsize.png"), dpi=300)
    plt.show()
    
    optimal_ws = summary_df.loc[summary_df['Mean_RMSE'].idxmin()]['Window_Size']
    print(f"\n>>> Based on Mean Validation RMSE, the Optimal Window Size is: {int(optimal_ws)} <<<")
else:
    print("No summary metrics available to analyze.")


No summary metrics available to analyze.
